# 01 — Data Quality Analysis

**Project:** Ontology-Guided Hypothesis Generation Using LLMs and Topic Modeling in mHealth Research

This notebook performs comprehensive data quality checks on the merged and cleaned dataset.

**Date:** 2026-08-29

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Paths
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_FILE = os.path.join(PROJECT_ROOT, 'data', 'processed', 'mhealth_final_cleaned_dataset.csv')
FIGURES_DIR = os.path.join(PROJECT_ROOT, 'reports', 'figures')
SUMMARIES_DIR = os.path.join(PROJECT_ROOT, 'reports', 'summaries')
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(SUMMARIES_DIR, exist_ok=True)

# Style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.2)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 150

print('Setup complete.')

Setup complete.


In [2]:
# Load dataset
df = pd.read_csv(DATA_FILE)
print(f'Total papers: {len(df)}')
print(f'Columns: {df.columns.tolist()}')
print(f'\nShape: {df.shape}')
df.head(3)

Total papers: 2628
Columns: ['pmid', 'title', 'abstract', 'year', 'mesh_terms', 'journal', 'authors', 'doi', 'document', 'cleaned_text', 'source']

Shape: (2628, 11)


,pmid,title,abstract,year,mesh_terms,journal,authors,doi,document,cleaned_text,source
0,31763204,The inaugural Qatar Critical Care Conference w...,Dr. Ibrahim Fawzy Hassan Local Host and QCCC 2...,2019.0,NaN,Qatar medical journal,Hassan IF; Alinier G,10.5339/qmj.2019.qccc.1,The inaugural Qatar Critical Care Conference w...,the inaugural qatar critical care conference w...,expanded
1,39776481,Benchmarking the clinical outcomes of Healthen...,BACKGROUND: Software as a Medical Device (SaMD...,2024.0,Humans; Chronic Disease/therapy; *Telemedicine...,Frontiers in public health,Kyriazakos S; Pnevmatikakis A; Kostopoulou K; ...,10.3389/fpubh.2024.1488687,Benchmarking the clinical outcomes of Healthen...,benchmarking the clinical outcomes of healthen...,expanded
2,42444014,Embedding community engagement in tuberculosis...,BACKGROUND: According to the WHO Global Tuberc...,NaN,NaN,NaN,NaN,NaN,Embedding community engagement in tuberculosis...,embedding community engagement in tuberculosis...,original


## 1. Missing Values Analysis

In [3]:
# Missing values per column
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

# Also count empty strings
empty_strings = df.apply(lambda col: (col.astype(str).str.strip() == '').sum())
empty_pct = (empty_strings / len(df) * 100).round(2)

quality_df = pd.DataFrame({
    'Column': df.columns,
    'Null Count': missing.values,
    'Null %': missing_pct.values,
    'Empty String Count': empty_strings.values,
    'Empty %': empty_pct.values,
    'Total Missing': (missing + empty_strings).values,
    'Total Missing %': ((missing + empty_strings) / len(df) * 100).round(2).values
})

print('=== MISSING VALUES ANALYSIS ===')
print(quality_df.to_string(index=False))

=== MISSING VALUES ANALYSIS ===
      Column  Null Count  Null %  Empty String Count  Empty %  Total Missing  Total Missing %
        pmid           0    0.00                   0      0.0              0             0.00
       title           0    0.00                   0      0.0              0             0.00
    abstract           0    0.00                   0      0.0              0             0.00
        year         491   18.68                   0      0.0            491            18.68
  mesh_terms        1004   38.20                   0      0.0           1004            38.20
     journal         499   18.99                   0      0.0            499            18.99
     authors         493   18.76                   0      0.0            493            18.76
         doi         524   19.94                   0      0.0            524            19.94
    document           0    0.00                   0      0.0              0             0.00
cleaned_text           0    

## 2. Duplicate Analysis

In [4]:
# Check for duplicate PMIDs
dup_pmids = df[df.duplicated(subset=['pmid'], keep=False)]
print(f'Duplicate PMIDs: {len(dup_pmids)} rows ({df.duplicated(subset=["pmid"]).sum()} true duplicates)')

# Check for duplicate titles
df['title_lower'] = df['title'].str.lower().str.strip()
dup_titles = df[df.duplicated(subset=['title_lower'], keep=False)]
print(f'Duplicate titles: {len(dup_titles)} rows ({df.duplicated(subset=["title_lower"]).sum()} true duplicates)')

if len(dup_pmids) > 0:
    print('\nDuplicate PMID samples:')
    print(dup_pmids[['pmid', 'title']].head(10))

df = df.drop(columns=['title_lower'])

Duplicate PMIDs: 0 rows (0 true duplicates)
Duplicate titles: 0 rows (0 true duplicates)


## 3. Abstract Quality Analysis

In [5]:
# Papers without abstracts
no_abstract = df[df['abstract'].fillna('').str.strip() == '']
print(f'Papers without abstracts: {len(no_abstract)}')

# Abstract length analysis
df['abstract_len'] = df['abstract'].fillna('').str.len()
print(f'\nAbstract Length Statistics:')
print(df['abstract_len'].describe().round(1))

# Very short abstracts (< 200 chars)
short_abstracts = df[df['abstract_len'] < 200]
print(f'\nVery short abstracts (< 200 chars): {len(short_abstracts)}')
if len(short_abstracts) > 0:
    print('\nSamples of short abstracts:')
    for _, row in short_abstracts.head(5).iterrows():
        print(f'  PMID {row["pmid"]}: {row["abstract"][:100]}... (len={row["abstract_len"]})')

Papers without abstracts: 0

Abstract Length Statistics:
count    2628.0
mean     1713.3
std       675.6
min       113.0
25%      1279.8
50%      1631.0
75%      2038.8
max      8990.0
Name: abstract_len, dtype: float64

Very short abstracts (< 200 chars): 5

Samples of short abstracts:
  PMID 31984208: An action plan prepared by @EuroRespSoc Group 01.04 (m-health/e-health) concerning the implementatio... (len=180)
  PMID 36675772: Given that dental practice is currently based on the "average" patient, providing therapeutic and re... (len=165)
  PMID 40218099: In the era of rapid technological advancement, healthcare is undergoing a profound transformation dr... (len=132)
  PMID 41072922: We outlined the characteristics and health status of individuals using a mobile health app, based on... (len=128)
  PMID 35624623: Humans have searched far beyond our planet to understand the fundamental principles and mechanisms o... (len=113)


In [6]:
# Abstract length distribution plot
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(df['abstract_len'], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(df['abstract_len'].median(), color='red', linestyle='--', label=f'Median: {df["abstract_len"].median():.0f}')
ax.axvline(200, color='orange', linestyle='--', label='Short threshold (200)')
ax.set_xlabel('Abstract Length (characters)')
ax.set_ylabel('Number of Papers')
ax.set_title('Distribution of Abstract Lengths')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'abstract_length_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

Figure saved.


C:\Users\VICTUS\AppData\Local\Temp\ipykernel_12860\206596514.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Publication Year Distribution

In [7]:
# Year distribution
df['year'] = df['year'].astype(str)
year_counts = df['year'].value_counts().sort_index()
print('=== YEAR DISTRIBUTION ===')
for year, count in year_counts.items():
    print(f'  {year}: {count} papers ({100*count/len(df):.1f}%)')

# Papers with invalid/missing years
invalid_years = df[~df['year'].str.match(r'^\d{4}$', na=False)]
print(f'\nPapers with invalid/missing years: {len(invalid_years)}')

=== YEAR DISTRIBUTION ===
  2012.0: 6 papers (0.2%)
  2018.0: 107 papers (4.1%)
  2019.0: 138 papers (5.3%)
  2020.0: 212 papers (8.1%)
  2021.0: 233 papers (8.9%)
  2022.0: 316 papers (12.0%)
  2023.0: 349 papers (13.3%)
  2024.0: 366 papers (13.9%)
  2025.0: 284 papers (10.8%)
  2026.0: 126 papers (4.8%)

Papers with invalid/missing years: 2628


In [8]:
# Year distribution plot
fig, ax = plt.subplots(figsize=(12, 5))
valid_years = year_counts[year_counts.index.str.match(r'^\d{4}$')]
colors = sns.color_palette('viridis', len(valid_years))
bars = ax.bar(valid_years.index, valid_years.values, color=colors, edgecolor='white')
ax.set_xlabel('Publication Year')
ax.set_ylabel('Number of Papers')
ax.set_title('Publication Year Distribution of mHealth Research Papers')

# Add value labels
for bar, val in zip(bars, valid_years.values):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
            str(val), ha='center', va='bottom', fontsize=9)

plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'year_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

Figure saved.


C:\Users\VICTUS\AppData\Local\Temp\ipykernel_12860\1851950966.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Journal/Source Analysis

In [9]:
# Journal frequency
if 'journal' in df.columns:
    journals = df['journal'].fillna('').str.strip()
    journals = journals[journals != '']
    print(f'Papers with journal info: {len(journals)} / {len(df)}')
    print(f'\nTop 15 Journals:')
    top_journals = journals.value_counts().head(15)
    for journal, count in top_journals.items():
        print(f'  {journal}: {count}')
else:
    print('Journal column not available in this dataset.')

Papers with journal info: 2129 / 2628

Top 15 Journals:
  Sensors (Basel, Switzerland): 114
  Studies in health technology and informatics: 73
  JMIR mHealth and uHealth: 60
  Journal of medical Internet research: 55
  Digital health: 45
  JMIR formative research: 42
  Telemedicine journal and e-health : the official journal of the American Telemedicine Association: 42
  International journal of environmental research and public health: 39
  Healthcare (Basel, Switzerland): 31
  Frontiers in public health: 26
  ACS applied materials & interfaces: 22
  Scientific reports: 22
  Biosensors: 21
  NPJ digital medicine: 20
  JMIR human factors: 19


## 6. MeSH Term Availability

In [10]:
# MeSH term coverage
mesh_available = df['mesh_terms'].fillna('').str.strip() != ''
print(f'Papers WITH MeSH terms: {mesh_available.sum()} ({100*mesh_available.sum()/len(df):.1f}%)')
print(f'Papers WITHOUT MeSH terms: {(~mesh_available).sum()} ({100*(~mesh_available).sum()/len(df):.1f}%)')

# Top MeSH terms
all_mesh = []
for terms in df['mesh_terms'].dropna():
    if str(terms).strip():
        for term in str(terms).split(';'):
            cleaned_term = term.strip().lstrip('*')
            if cleaned_term:
                all_mesh.append(cleaned_term)

mesh_series = pd.Series(all_mesh)
print(f'\nTotal MeSH term occurrences: {len(mesh_series)}')
print(f'Unique MeSH terms: {mesh_series.nunique()}')
print(f'\nTop 20 MeSH terms:')
print(mesh_series.value_counts().head(20).to_string())

Papers WITH MeSH terms: 1624 (61.8%)
Papers WITHOUT MeSH terms: 1004 (38.2%)

Total MeSH term occurrences: 13771
Unique MeSH terms: 3140

Top 20 MeSH terms:
Humans                         1534
Telemedicine                    554
Mobile Applications             542
Female                          516
Male                            441
Adult                           339
Middle Aged                     292
Smartphone                      242
Aged                            237
Wearable Electronic Devices     213
Digital Health                  153
Surveys and Questionnaires      141
Adolescent                      111
Young Adult                     107
Delivery of Health Care          77
Cross-Sectional Studies          76
Feasibility Studies              76
Child                            75
Qualitative Research             73
Artificial Intelligence          67


## 7. Document Length Distribution

In [11]:
# Document length (title + abstract)
df['doc_len'] = df['document'].fillna('').str.len()
df['doc_word_count'] = df['document'].fillna('').str.split().str.len()

print('=== DOCUMENT LENGTH STATISTICS ===')
print('\nCharacter length:')
print(df['doc_len'].describe().round(1))
print('\nWord count:')
print(df['doc_word_count'].describe().round(1))

# Suspicious records (very short documents)
very_short = df[df['doc_word_count'] < 50]
print(f'\nVery short documents (< 50 words): {len(very_short)}')
if len(very_short) > 0:
    for _, row in very_short.head(5).iterrows():
        print(f'  PMID {row["pmid"]}: "{row["title"][:80]}..." ({row["doc_word_count"]} words)')

=== DOCUMENT LENGTH STATISTICS ===

Character length:
count    2628.0
mean     1823.6
std       690.3
min       158.0
25%      1376.8
50%      1742.0
75%      2158.8
max      9107.0
Name: doc_len, dtype: float64

Word count:
count    2628.0
mean      252.8
std        96.6
min        22.0
25%       190.0
50%       243.0
75%       298.0
max      1400.0
Name: doc_word_count, dtype: float64

Very short documents (< 50 words): 14
  PMID 36433218: "Advances in E-Health and Mobile Health Monitoring...." (49 words)
  PMID 37334840: "Snoring Patterns During Hypoglossal Nerve Stimulation Therapy Up-Titration...." (47 words)
  PMID 37203567: "Requirements to mHealth Applications for Animal Owners: A Narrative Review...." (49 words)
  PMID 42489000: "The Next Generation of Wearables Won't Need the Cloud...." (49 words)
  PMID 41646838: "Equitable access to oncology clinical trials: harnessing technology to reduce ge..." (44 words)


## 8. Data Quality Summary Report

In [12]:
# Generate comprehensive summary
report = []
report.append('=' * 60)
report.append('DATA QUALITY ANALYSIS REPORT')
report.append('=' * 60)
report.append(f'\nDataset: {DATA_FILE}')
report.append(f'Total papers: {len(df)}')
report.append(f'Columns: {df.columns.tolist()}')
report.append(f'\n--- Missing Values ---')
report.append(quality_df.to_string(index=False))
report.append(f'\n--- Duplicates ---')
report.append(f'Duplicate PMIDs: {df.duplicated(subset=["pmid"]).sum()}')
report.append(f'\n--- Abstract Quality ---')
report.append(f'Papers without abstracts: {len(no_abstract)}')
report.append(f'Very short abstracts (<200 chars): {len(short_abstracts)}')
report.append(f'Mean abstract length: {df["abstract_len"].mean():.0f} chars')
report.append(f'\n--- Year Coverage ---')
for year, count in year_counts.items():
    report.append(f'  {year}: {count}')
report.append(f'\n--- MeSH Terms ---')
report.append(f'With MeSH: {mesh_available.sum()} ({100*mesh_available.sum()/len(df):.1f}%)')
report.append(f'Without MeSH: {(~mesh_available).sum()} ({100*(~mesh_available).sum()/len(df):.1f}%)')
report.append(f'\n--- Document Quality ---')
report.append(f'Mean word count: {df["doc_word_count"].mean():.0f}')
report.append(f'Very short docs (<50 words): {len(very_short)}')
report.append(f'\n--- OVERALL ASSESSMENT ---')
report.append(f'Dataset is in good condition for downstream processing.')
issues = []
if len(no_abstract) > 0: issues.append(f'{len(no_abstract)} papers without abstracts')
if df.duplicated(subset=['pmid']).sum() > 0: issues.append(f'{df.duplicated(subset=["pmid"]).sum()} duplicate PMIDs')
if len(very_short) > 0: issues.append(f'{len(very_short)} very short documents')
if issues:
    report.append(f'Issues found: {"; ".join(issues)}')
else:
    report.append('No critical issues found.')

report_text = '\n'.join(report)

# Save report
report_path = os.path.join(SUMMARIES_DIR, 'data_quality_report.txt')
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report_text)

print(report_text)
print(f'\nReport saved to: {report_path}')

DATA QUALITY ANALYSIS REPORT

Dataset: E:\MAJOR PROJECT\data\processed\mhealth_final_cleaned_dataset.csv
Total papers: 2628
Columns: ['pmid', 'title', 'abstract', 'year', 'mesh_terms', 'journal', 'authors', 'doi', 'document', 'cleaned_text', 'source', 'abstract_len', 'doc_len', 'doc_word_count']

--- Missing Values ---
      Column  Null Count  Null %  Empty String Count  Empty %  Total Missing  Total Missing %
        pmid           0    0.00                   0      0.0              0             0.00
       title           0    0.00                   0      0.0              0             0.00
    abstract           0    0.00                   0      0.0              0             0.00
        year         491   18.68                   0      0.0            491            18.68
  mesh_terms        1004   38.20                   0      0.0           1004            38.20
     journal         499   18.99                   0      0.0            499            18.99
     authors         

In [13]:
# Clean up temporary columns
df = df.drop(columns=['abstract_len', 'doc_len', 'doc_word_count'], errors='ignore')
print('Data quality analysis complete.')

Data quality analysis complete.
